In [ ]:
%load_ext autoreload
%autoreload 2
%config Completer.use_jedi = False

In [ ]:
import numpy as np
import pickle
import os, shutil, yaml

from matplotlib import pyplot as plt
from pylab import *

from tqdm import tqdm

In [ ]:
from pymatgen.io.ase import AseAtomsAdaptor

In [ ]:
import torch
import torch.nn as nn

In [ ]:
import sys
sys.path.append('./code')

In [ ]:
from data import XASGraphDataset 
from models import XASLightningModule 

In [ ]:
from captum.attr import IntegratedGradients

import dgl
from matgl.utils.cutoff import polynomial_cutoff
from matgl.graph.compute import (
    compute_theta_and_phi,
    create_line_graph,
)

In [ ]:
GRID=np.linspace(8900, 9050, 1000)
energy_grid = GRID[333:733:2]
nblocks, cutoff, threebody_cutoff = 3, 4.0, 4.0
r1, r2, rZn = 3.0, 4.5, 4.2

In [ ]:
ROOT_PATH = './'

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
with open(ROOT_PATH + "/configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

In [ ]:
element_map = ["O", "H", "Zn", "Cl"]

### 1. Load data

In [ ]:
structures = pickle.load(open(ROOT_PATH + '/dataset/all_structures.pkl', 'rb'))
spectra = torch.load(ROOT_PATH + "/dataset/all_spectra.pt")

In [ ]:
all_data = XASGraphDataset(structures, spectra, cutoff=cutoff)

### 2. Integrated Gradients

#### 2.0 Define functions

In [ ]:
def predict(mlp_model, data_loader_feat):
    predictions = []
    val_loss = 0.0
    
    with torch.no_grad():   
        for feats, spectra in data_loader_feat:
            feats, spectra = feats.to(device), spectra.to(device)
            y_hat = mlp_model(feats)
    
            predictions.append(y_hat.cpu())
    
            loss = torch.nn.MSELoss()(y_hat, spectra)
            val_loss += loss.item() * spectra.size(0) 
    
    predictions = torch.cat(predictions, dim=0)
    
    print(f"Loss: {val_loss / len(val_loader_feat.dataset)}")    
    return val_loss / len(val_loader_feat.dataset), predictions

In [ ]:
def plot_importance(importance): 
    plt.figure(figsize=(12,4))
    plt.bar(range(len(importance)), importance, color=color_list)
    for i, elem in enumerate(elements):
        plt.text(i, 0.2, elem, ha="center", fontsize=8)
        plt.text(i, 0.05, i+1, ha="center", fontsize=7)
    plt.xlabel("Atom index")
    plt.ylabel("Importance (|IG| sum)")
    plt.ylim(0, 0.8)

In [ ]:
# ===========================================================
# Define node wrapper model
# ===========================================================

class NodeEmbeddingWrapper(torch.nn.Module):
    """Wraps model from node embeddings → spectrum."""
    def __init__(self, model, edge_feat):
        super().__init__()
        self.model = model
        self.edge_feat = edge_feat
        self.state_feat = None
        self.energy_idx = None

    def forward(self, node_feat, g, state_attr=None):
        # Run rest of GNN manually, reusing intermediate features
        edge_feat = self.edge_feat
        state_feat = self.state_feat

        # --- build proper 3-body line graph ---

        l_g = create_line_graph(g, self.model.gnn.threebody_cutoff)
        l_g.apply_edges(compute_theta_and_phi)
        three_body_basis = self.model.gnn.basis_expansion(l_g)
        three_body_cutoff = polynomial_cutoff(g.edata["bond_dist"], self.model.gnn.threebody_cutoff)
        

        # --- message passing loop ---
        for i in range(self.model.gnn.nblocks):
            edge_feat = self.model.gnn.three_body_interactions[i](
                g, l_g, three_body_basis, three_body_cutoff, node_feat, edge_feat
            )
            edge_feat, node_feat, state_feat = self.model.gnn.graph_layers[i](
                g, edge_feat, node_feat, state_feat
            )

        # Extract absorber features (first atom)
        num_atoms_per_graph = g.batch_num_nodes().tolist()
        first_atom_idx = torch.cumsum(
            torch.tensor([0] + num_atoms_per_graph[:-1], device=node_feat.device),
            dim=0
        )
        absorber_feats = node_feat[first_atom_idx]
        spectra = self.model.spectrum_head(absorber_feats)
        return spectra[:, self.energy_idx]

In [ ]:
# ===========================================================
# Define bond length wrapper model
# ===========================================================

class BondDistanceAttributionWrapper(torch.nn.Module):
    """Forward from RBF edge features → spectrum intensity."""
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.state_feat = None
        self.energy_idx = None

    def forward(self, bond_dist, g, state_attr=None):
        
        node_types = g.ndata["node_type"]
        expanded_dists = self.model.gnn.bond_expansion(bond_dist)
        g.edata["rbf"] = expanded_dists
        
        node_feat, edge_feat, state_feat = self.model.gnn.embedding(node_types, expanded_dists, state_attr=None)
    
        # Build 3-body line graph
        l_g = create_line_graph(g, self.model.gnn.threebody_cutoff)
        l_g.apply_edges(compute_theta_and_phi)
        three_body_basis = self.model.gnn.basis_expansion(l_g)
        three_body_cutoff = polynomial_cutoff(
            g.edata["bond_dist"], self.model.gnn.threebody_cutoff
        )

        for i in range(self.model.gnn.nblocks):
            edge_feat = self.model.gnn.three_body_interactions[i](
                g, l_g, three_body_basis, three_body_cutoff, node_feat, edge_feat
            )
            edge_feat, node_feat, state_feat = self.model.gnn.graph_layers[i](
                g, edge_feat, node_feat, state_feat
            )

        # Absorber → spectrum
        num_atoms_per_graph = g.batch_num_nodes().tolist()
        first_atom_idx = torch.cumsum(
            torch.tensor([0] + num_atoms_per_graph[:-1], device=node_feat.device),
            dim=0,
        )
        absorber_feats = node_feat[first_atom_idx]
        spectra = self.model.spectrum_head(absorber_feats)
        return spectra[:, self.energy_idx]


In [ ]:
# ===========================================================
# Node-wise attribution
# ===========================================================

def calc_importance(energy_idx, baseline, nonneg=False): 
    # ===========================================================
    # Integrated Gradients attribution
    # ===========================================================
    
    wrapper = NodeEmbeddingWrapper(model, edge_feat)
    wrapper.energy_idx = energy_idx
    
    ig = IntegratedGradients(wrapper)
    
    attributions, delta = ig.attribute(node_feat, baselines=baseline, 
                                       additional_forward_args=(g,),  
                                       return_convergence_delta=True, 
                                       internal_batch_size=1
                                      )
    
    # ===========================================================
    # Aggregate per-atom importance
    # ===========================================================
    
    if nonneg: 
        importance = attributions.abs().sum(dim=1).detach().cpu().numpy()
    else: 
        importance = attributions.sum(dim=1).detach().cpu().numpy()  

    return importance

In [ ]:
# ===========================================================
# bond length attribution
# ===========================================================

def calc_importance_bond_dist(energy_idx, baseline): 
    
    wrapper = BondDistanceAttributionWrapper(model) 
    wrapper.energy_idx = energy_idx
    
    ig = IntegratedGradients(wrapper)
    
    attributions, delta = ig.attribute(bond_dist, baselines=baseline, 
                                       additional_forward_args=(g,), 
                                       return_convergence_delta=True, 
                                       internal_batch_size=1
                                      )
    
    importance = attributions.squeeze().detach().cpu().numpy()

    return importance

#### 2.1. Load model

In [ ]:
gnn_config = dict(
    nblocks = config['gnn']['nblocks'], 
    cutoff = config['gnn']['cutoff'], 
    threebody_cutoff = config['gnn']['threebody_cutoff']
)

head_config = dict(
    hidden_dims = config['head']['hidden_dims'], 
    output_size = config['head']['output_size'], 
    drop_rate = config['head']['drop_rate'], 
)

model = XASLightningModule(gnn_config, head_config, learning_rate=config['training']['lr'])

model.load_state_dict(torch.load(os.path.join('./model', "full_model.pth"), weights_only=True))

model = model.to(device)
model.eval()

#### 2.2. Check prediction

In [ ]:
cluster_reps = [1955, 923, 1878, 1210, 1858, 976]

In [ ]:
cluster_idx = 0
sp_idx = cluster_reps[cluster_idx]
structure = structures[sp_idx]

In [ ]:
g = all_data[sp_idx][0].to("cuda")
pred_spectrum_cpu = model(g)[0].detach().cpu().numpy()
plt.plot(energy_grid, spectra[sp_idx])
plt.plot(energy_grid, pred_spectrum_cpu)

#### 2.3 Get node (0 baseline and CF) and bond-length attributions

In [ ]:
# ===========================================================
# Node attribution, 0 baseline 
# ===========================================================

for cluster_idx in range(1,2): 
    sp_idx = cluster_reps[cluster_idx]
    structure = structures[sp_idx]
    g = all_data[sp_idx][0].to("cuda")
    
    node_types = g.ndata["node_type"].cpu().numpy().flatten()
    bond_dist = g.edata["bond_dist"].clone().detach().requires_grad_(True)
    elements = [element_map[int(i)] for i in node_types]
    
    # # -------- Get node attributions ------------ ##
    with torch.no_grad():
        node_types = g.ndata["node_type"]
        expanded_dists = model.gnn.bond_expansion(g.edata["bond_dist"])
    
        g.edata["rbf"] = expanded_dists
        
        node_feat, edge_feat, state_feat = model.gnn.embedding(
            node_types, expanded_dists, state_attr=None
        )   
        
    baseline = torch.zeros_like(node_feat)  # simple neutral baseline
    
    importance_arr = np.zeros((200, node_feat.shape[0]))
    for energy_idx in tqdm(np.arange(0, 200, 1)):
        importance_arr[energy_idx] = calc_importance(energy_idx, baseline, nonneg=True)
    
    np.savetxt('./data/data-IG/0_baseline-importance_node-structure_%s-%s-%s-%s.txt'%(sp_idx, nblocks, cutoff, threebody_cutoff), importance_arr)

In [ ]:
# ===========================================================
# Node attribution, (O+Cl)/2 baseline 
# ===========================================================

for cluster_idx in range(1,2): 
    sp_idx = cluster_reps[cluster_idx]
    structure = structures[sp_idx]
    g = all_data[sp_idx][0].to("cuda")
    
    node_types = g.ndata["node_type"].cpu().numpy().flatten()
    bond_dist = g.edata["bond_dist"].clone().detach().requires_grad_(True)
    elements = [element_map[int(i)] for i in node_types]
    
    # # -------- Get node attributions ------------ ##
    
    with torch.no_grad():
        node_types = g.ndata["node_type"]
        expanded_dists = model.gnn.bond_expansion(g.edata["bond_dist"])
    
        g.edata["rbf"] = expanded_dists
        
        node_feat, edge_feat, state_feat = model.gnn.embedding(
            node_types, expanded_dists, state_attr=None
        )   
    
    # node_feat is (N_atoms, dim_node_embedding)
    node_feat = node_feat.clone().detach().requires_grad_(True)
    
    ### Get baseline from averaged embedding
    elem_embs = torch.stack([
        node_feat[node_types==0][0].detach(), 
        node_feat[node_types==3][0].detach(), 
    ])
    mean_elem_emb = elem_embs.mean(dim=0, keepdim=True)  # (1, 64)
    baseline = mean_elem_emb.expand(g.num_nodes(), -1).clone().to(g.device)
    
    importance_arr = np.zeros((200, node_feat.shape[0]))
    for energy_idx in tqdm(np.arange(0, 200, 1)):
        importance_arr[energy_idx] = calc_importance(energy_idx, baseline)
    
    np.savetxt('./data/data-IG/OCl_baseline-importance_node-structure_%s-%s-%s-%s.txt'%(sp_idx, nblocks, cutoff, threebody_cutoff), importance_arr)

In [ ]:
# ===========================================================
# Bond length attribution 
# ===========================================================
for cluster_idx in range(1,2): 
    sp_idx = cluster_reps[cluster_idx]
    structure = structures[sp_idx]
    g = all_data[sp_idx][0].to("cuda")
    
    node_types = g.ndata["node_type"].cpu().numpy().flatten()
    bond_dist = g.edata["bond_dist"].clone().detach().requires_grad_(True)
    elements = [element_map[int(i)] for i in node_types]
    
    with torch.no_grad():
        node_types = g.ndata["node_type"]
        expanded_dists = model.gnn.bond_expansion(g.edata["bond_dist"])
    
        g.edata["rbf"] = expanded_dists
        
        node_feat, edge_feat, state_feat = model.gnn.embedding(
            node_types, expanded_dists, state_attr=None
        )   

    # -------- Get bond length attributions ------------ ##
    
    delta = 0.05  # Effect of bond shortening; Compare current to longer bonds; The negative of attribution is the effect of bond elongation
    baseline = (g.edata["bond_dist"] * (1 + delta)).clone().detach()
    
    importance_arr = np.zeros((200, bond_dist.shape[0]))
    for energy_idx in tqdm(np.arange(0, 200, 1)):
        importance_arr[energy_idx] = calc_importance_bond_dist(energy_idx, baseline)
    
    np.savetxt('./data/data-IG/%s-importance_bond-structure_%s-%s-%s-%s.txt'%(delta, sp_idx, nblocks, cutoff, threebody_cutoff), importance_arr)

#### 2.5 Plot 

In [ ]:
cluster_idx = 0
sp_idx = cluster_reps[cluster_idx]
structure = structures[sp_idx]

In [ ]:
g = all_data[sp_idx][0].to("cuda")
pred_spectrum_cpu = model(g)[0].detach().cpu().numpy()

In [ ]:
node_importance_0baseline_arr = np.loadtxt('./data/0_baseline-importance_node-structure_%s-%s-%s-%s.txt'%(sp_idx, nblocks, cutoff, threebody_cutoff))
node_importance_OCl_arr = np.loadtxt('./data/OCl_baseline-importance_node-structure_%s-%s-%s-%s.txt'%(sp_idx, nblocks, cutoff, threebody_cutoff))

delta = 0.05   
edge_importance_arr = np.loadtxt('./data/%s-importance_bond-structure_%s-%s-%s-%s.txt'%(delta, sp_idx, nblocks, cutoff, threebody_cutoff))

##### 2.5.1 Plot node-wise attribution

In [ ]:
distance_arr = np.zeros(len(structure))
for ii in range(len(distance_arr)): 
    distance_arr[ii] = structure.get_distance(0, ii)

color_list = []
for d in distance_arr: 
    if d <= r1: 
        color_list.append('r')
    elif d <= rZn: 
        color_list.append('orange')
    elif d <= r2: 
        color_list.append('g')
    elif d <= 5.0: 
        color_list.append('b')
    else:
        color_list.append('gray')

In [ ]:
atoms = AseAtomsAdaptor.get_atoms(structure)
first_neighbors = structure.get_neighbors(structure[0], r=r1)  # Using 3.0 Angstrom cutoff
first_neighbors = [neighbor for neighbor in first_neighbors if neighbor.specie.symbol != 'H']
first_neighbor_species = [neighbor.specie.symbol for neighbor in first_neighbors]

# Get Zn neighbors
Zn_neighbors = structure.get_neighbors(structure[0], r=rZn)  
Zn_neighbors = [neighbor for neighbor in Zn_neighbors if neighbor.specie.symbol == 'Zn']
Zn_neighbors_species = [neighbor.specie.symbol for neighbor in Zn_neighbors]

idxs_Cl = []
idxs_O = []
for site in first_neighbors: 
    if site.specie.symbol == 'Cl': 
        idxs_Cl.append(site.index)
    if site.specie.symbol == 'O': 
        idxs_O.append(site.index)
idxs_Zn = []
for site in Zn_neighbors: 
    idxs_Zn.append(site.index)

idxs_Cl.sort()        
idxs_O.sort()        
idxs_Zn.sort()        


In [ ]:
mean_attr = abs(node_importance_0baseline_arr).mean(axis=0)
mean_attr_nmlz = mean_attr / np.max(mean_attr)
plot_importance(mean_attr_nmlz)
plt.ylim(-0.1, 1.1)
# plt.savefig('./figures/Node_attr-cluster_%s-structure_%s.png'%(cluster_idx, sp_idx), bbox_inches='tight', dpi=100)

In [ ]:
plt.plot(energy_grid, spectra[sp_idx], label='DFT', color='gray')
plt.plot(energy_grid, pred_spectrum_cpu, label='ML', linestyle='--')

for idx in idxs_Cl + idxs_O:
    element = structure[idx].specie.symbol
    plt.plot(energy_grid, abs(node_importance_0baseline_arr[:,idx])/np.max(abs(node_importance_0baseline_arr[:,1:])), 
             alpha=1, label=element+",%s"%(idx+1))
plt.axvline(x=energy_grid[30], linestyle=':', color='k', alpha=0.8, lw=1)  ## 8959 eV
plt.axvline(x=energy_grid[43], linestyle=':', color='k', alpha=0.8, lw=1)  ## 8963 eV
plt.legend()
plt.title("FSS, 0 baseline")
plt.ylim(-0.2, 2.7)
plt.xlim(energy_grid[0], energy_grid[-1])
# plt.savefig('./figures/Node_FSS-0baseline-cluster_%s-structure_%s.png'%(cluster_idx, sp_idx), bbox_inches='tight', dpi=100)

In [ ]:
plt.plot(energy_grid, spectra[sp_idx], label='DFT', color='gray')
plt.plot(energy_grid, pred_spectrum_cpu, label='ML', linestyle='--')

for idx in idxs_Cl + idxs_O:
    element = structure[idx].specie.symbol
    plt.plot(energy_grid, node_importance_OCl_arr[:,idx]/np.max(abs(node_importance_OCl_arr[:,1:])), 
             alpha=1, label=element+",%s"%(idx+1))
plt.axvline(x=energy_grid[30], linestyle=':', color='k', alpha=0.8, lw=1)  ## 8959 eV
plt.axvline(x=energy_grid[43], linestyle=':', color='k', alpha=0.8, lw=1)  ## 8963 eV
plt.legend()
plt.title("FSS, CF")
plt.ylim(-1, 2.7)
plt.xlim(energy_grid[0], energy_grid[-1])

##### 2.5.2. Plot bond length attribution

In [ ]:
### Classify bonds 

src, dst = g.edges()
bond_dist = g.edata["bond_dist"].cpu().numpy().flatten()
node_types = g.ndata["node_type"].cpu().numpy()

fs_bonds = []
mask_in = (dst == 0)
for bond_idx in range(len(bond_dist)): 
    if mask_in[bond_idx]: 
        if node_types[src[bond_idx]] == 1: # ignore H atom
            continue
        fs_bonds.append(
            (bond_idx, src[bond_idx].item())
        )
fs_bonds.sort(key=lambda x: x[1])

ss_bonds = []
for _, src_idx in fs_bonds: 
    dst_idx, dst_element = src_idx, node_types[src_idx]
    mask_in = (dst == dst_idx)
    for bond_idx in range(len(bond_dist)): 
        if mask_in[bond_idx]: 
            if node_types[src[bond_idx]] == 1: # ignore H atom
                continue
            if src[bond_idx] != 0: 
                ss_bonds.append(
                    (bond_idx, src[bond_idx].item(), dst[bond_idx].item(), 
                     structure.get_distance(0, src[bond_idx].item()), structure.get_distance(0, dst[bond_idx].item()))
                )

In [ ]:
plt.plot(energy_grid, spectra[sp_idx], label='DFT', color='gray')
plt.plot(energy_grid, pred_spectrum_cpu, label='ML', linestyle='--')
for bond_idx, src_idx in fs_bonds: 
    element_symbol, bond_len = element_map[node_types[src_idx]], bond_dist[bond_idx]
    if bond_len <= r1: 
        plt.plot(energy_grid, -edge_importance_arr[:, bond_idx]/np.max(abs(edge_importance_arr)),
                 label=f'{element_symbol},{src_idx+1}, {bond_len:.2f}Å')
plt.axvline(x=energy_grid[30], linestyle=':', color='k', alpha=0.8, lw=1)  ## 8959 eV
plt.axvline(x=energy_grid[43], linestyle=':', color='k', alpha=0.8, lw=1)  ## 8963 eV
plt.legend(loc=(1.02, 0.1))
plt.title("∂ spectrum / ∂ bond_len")
plt.ylim(-1, 2.5)
plt.xlim(energy_grid[0], energy_grid[-1])
# plt.savefig('./figures/Edge_FSS-cluster_%s-structure_%s.png'%(cluster_idx, sp_idx), bbox_inches='tight', dpi=100)